In [ ]:
import pandas as pd
import numpy as np
import os

SIM_PATH = 'output/embeddings/task2_similarity.csv' 
SENT_PATH = 'output/recommendations.csv'

if not os.path.exists(SIM_PATH):
    SIM_PATH = '../' + SIM_PATH
    SENT_PATH = '../' + SENT_PATH

try:
    df_sim_long = pd.read_csv(SIM_PATH)
    if 'product' in str(df_sim_long.columns[0]).lower():
        df_sim_long = pd.read_csv(SIM_PATH, skiprows=1, header=None)

    sentiment_df = pd.read_csv(SENT_PATH)
except Exception as e:
    print(f"Load Error: {e}")

def get_numeric_col(df):
    for i in range(len(df.columns)-1, -1, -1):
        try:
            pd.to_numeric(df.iloc[:, i].head(10))
            return i
        except:
            continue
    return 2 

score_idx = get_numeric_col(df_sim_long)
id1_idx = score_idx - 2
id2_idx = score_idx - 1

sent_col = 'avg_sentiment' if 'avg_sentiment' in sentiment_df.columns else 'sentiment_score'
sent_map = dict(zip(sentiment_df['productId'].astype(str).str.strip().str.lower(), sentiment_df[sent_col]))

def get_hybrid_recommendations(product_id, top_n=5):
    product_id = str(product_id).strip()
    
    c1 = df_sim_long.iloc[:, id1_idx].astype(str).str.strip()
    c2 = df_sim_long.iloc[:, id2_idx].astype(str).str.strip()
    

    matches = df_sim_long[(c1 == product_id) | (c2 == product_id)].drop_duplicates().copy()

    hybrid_results = []
    for _, row in matches.iterrows():
        p_a, p_b = str(row.iloc[id1_idx]).strip(), str(row.iloc[id2_idx]).strip()
        other_pid = p_b if p_a == product_id else p_a
        
        if other_pid == product_id or "product" in other_pid.lower(): 
            continue

        try:
            sim_value = float(row.iloc[score_idx])
            sentiment_val = sent_map.get(other_pid.lower(), 0)
            normalized_sentiment = (sentiment_val + 1) / 2
            
            final_score = (sim_value * 0.7) + (normalized_sentiment * 0.3)
            hybrid_results.append((other_pid, final_score))
        except:
            continue

    return sorted(hybrid_results, key=lambda x: x[1], reverse=True)[:top_n]

all_ids = pd.concat([df_sim_long.iloc[:, id1_idx], df_sim_long.iloc[:, id2_idx]]).astype(str)
real_ids = all_ids[~all_ids.str.lower().str.contains('product|review|unnamed')].value_counts()

if not real_ids.empty:
    test_id = real_ids.index[0]
    print(f" Testing with: {test_id}")
    results = get_hybrid_recommendations(test_id)
    
    if results:
        print(f"\n Recommendations for {test_id}:")
        for pid, score in results:
            print(f" {pid} | Hybrid Score: {score:.4f}")
    else:
        print(f" No connections found for {test_id} in the repaired columns.")
else:
    print(" No valid Product IDs found. Check if the file is empty.")

 Testing with: B0051COPH6

 Recommendations for B0051COPH6:
 B0018KR8V0 | Hybrid Score: 0.7029
 B002QWP89S | Hybrid Score: 0.6248
 B003FDC2I2 | Hybrid Score: 0.5652
 B003FDC2I2 | Hybrid Score: 0.5652
 B0034EDLS2 | Hybrid Score: 0.5592
